# 구글 마운트

In [ ]:
from google.colab import drive
import os

def _drive_ready(mountpoint='/content/drive'):
    """진짜로 Drive 가 붙어 있는가.

    isdir('/content/drive/MyDrive') 만 보면 안 된다 — 마운트가 끊긴 자리에 같은 이름의
    폴더가 남으면 붙지도 않았는데 붙은 것처럼 보이고, 그 뒤 읽기·쓰기가 전부 조용히
    로컬 디스크로 새 나간다(한참 뒤 엉뚱한 데서 FileNotFoundError 로 터진다)."""
    return os.path.ismount(mountpoint) and any(
        os.path.isdir(os.path.join(mountpoint, n)) for n in ('MyDrive', 'My Drive')
    )


def _ensure_drive(mountpoint='/content/drive'):
    """이미 붙어 있으면 넘어가고, 아니면 붙인다. 마운트 자리에 찌꺼기가 남아
    mount 가 거부하는 경우("Mountpoint must not already contain files")까지 복구한다."""
    if _drive_ready(mountpoint):
        return
    try:
        drive.mount(mountpoint)
        return
    except ValueError:
        pass

    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if not os.path.ismount(mountpoint):
        # 마운트가 아니니 여기 있는 건 Drive 원본이 아니라 로컬 찌꺼기다. 지우지 말고
        # 옆으로 치워 둔다 — 이 상태에서 저장된 파일이 있었다면 살릴 수 있게.
        # (rm -rf 로 지우는 방법은 쓰지 않는다. 마운트가 살아 있었다면 원본이 날아간다.)
        stale, n = mountpoint + '_stale', 1
        while os.path.exists(stale):
            n += 1
            stale = f'{mountpoint}_stale{n}'
        os.rename(mountpoint, stale)
        print(f'마운트가 아닌 찌꺼기 폴더를 {stale} 로 치웠습니다.')
    drive.mount(mountpoint, force_remount=True)


_ensure_drive()

# 예외_아웃 열 추가

In [ ]:
import os
import re
import unicodedata
import pandas as pd


# ==================== 설정 ====================
YEAR = 2022
DRIVE_BASE = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"


# ── 파일 범위 지정 헬퍼 (3. 주자상황 체크.ipynb 와 동일한 규칙)
#   특정 파일까지만  : end_file='2022_07_a'
#   특정 파일부터    : start_file='2022_04_a'
#   특정 구간만      : start_file='2022_04_a', end_file='2022_07_c'
#   year 생략 시 start_file/end_file 앞 4자리(예: '2023_04_a' -> 2023)로 자동 판단
def _slice_file_range(source_files, start_file=None, end_file=None):
    """정렬된 source_files를 file stem 기준 [start_file, end_file] 구간만 남긴다.
    start_file 생략 -> 처음부터 / end_file 생략 -> 끝까지 (양끝 포함).
    '2022_07_a' / '2022_07_a.xlsx' 둘 다 허용."""
    if not source_files or (not start_file and not end_file):
        return source_files
    stems = [os.path.splitext(os.path.basename(p))[0] for p in source_files]

    def _idx(name, label):
        stem = os.path.splitext(name)[0]
        if stem not in stems:
            raise ValueError(f"{label}='{name}' 을(를) 파일 목록에서 찾을 수 없습니다")
        return stems.index(stem)

    i = _idx(start_file, 'start_file') if start_file else 0
    j = _idx(end_file, 'end_file') if end_file else len(stems) - 1
    if i > j:
        raise ValueError(f"범위 오류: start_file='{stems[i]}' 이 end_file='{stems[j]}' 보다 뒤입니다")
    return source_files[i:j + 1]


def _infer_year(start_file, end_file, fallback_year):
    """start_file/end_file 앞 4자리 숫자(YYYY_MM_x 형식)를 연도로 추정한다.
    둘 다 없거나 형식이 안 맞으면 fallback_year를 그대로 쓴다."""
    for name in (start_file, end_file):
        if name:
            m = re.match(r'^(\d{4})_', os.path.basename(name))
            if m:
                return int(m.group(1))
    return fallback_year


def _collect_pbp_files(year):
    """{DRIVE_BASE}/{year} 하위 'N월' 폴더들을 순서대로 훑어 PBP 파일 경로를 정렬된 리스트로 돌려준다."""
    year_dir = f"{DRIVE_BASE}/{year}"
    month_pat = re.compile(r'^\d+월$')
    file_pat  = re.compile(rf'^{year}_\d{{2}}_[a-z]+\.xlsx$')
    source_files = []
    for entry in sorted(os.listdir(year_dir)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(year_dir, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if file_pat.match(f_nfc):
                source_files.append(os.path.join(month_dir, f_nfc))
    source_files.sort()
    return source_files


# ==================== 예외_아웃 열 일괄 백필 ====================
def backfill_예외_아웃_column(year=None, start_file=None, end_file=None):
    """
    {DRIVE_BASE}/{year} 하위 월별 폴더(예: 3월/, 4월/...)의 PBP 엑셀 파일(YYYY_MM_x.xlsx) 중
    start_file~end_file 구간만(둘 다 생략 시 전체) 스캔해서 order와 batter 사이에 예외_아웃 열이 있도록 맞춘다.
      - year를 생략하면 start_file/end_file에 적힌 연도(앞 4자리)를 그대로 쓰고,
        둘 다 없으면 위 YEAR 기본값을 쓴다.
      - 이미 '예외_아웃' 열이 있으면 스킵
      - 과거에 'R예외_아웃'이라는 이름으로 추가된 열이 있으면 '예외_아웃'으로 이름만 변경
      - 둘 다 없으면 order 바로 다음에 빈 칸으로 새로 추가
    안전하게 여러 번 실행 가능.
    """
    year = year if year is not None else _infer_year(start_file, end_file, YEAR)
    year_dir = f"{DRIVE_BASE}/{year}"
    if not os.path.exists(year_dir):
        print(f"경로가 존재하지 않습니다: {year_dir}")
        return

    source_files = _collect_pbp_files(year)
    _n_all = len(source_files)
    source_files = _slice_file_range(source_files, start_file, end_file)
    if not source_files:
        print("해당 범위에 파일이 없습니다.")
        return
    if len(source_files) != _n_all:
        _s = os.path.splitext(os.path.basename(source_files[0]))[0]
        _e = os.path.splitext(os.path.basename(source_files[-1]))[0]
        print(f"[{year}년 / 범위 지정] {_s} ~ {_e} ({len(source_files)}/{_n_all}개 파일)\n")
    else:
        print(f"[{year}년 / 전체] {_n_all}개 파일\n")

    added_files = []
    renamed_files = []
    skipped_files = []
    error_files = []

    for fpath in source_files:
        fname_nfc = os.path.basename(fpath)
        try:
            df = pd.read_excel(fpath, engine="openpyxl")

            if '예외_아웃' in df.columns:
                skipped_files.append(fname_nfc)
                print(f"  [스킵] {fname_nfc}: 이미 열이 존재함")
                continue

            if 'R예외_아웃' in df.columns:
                df = df.rename(columns={'R예외_아웃': '예외_아웃'})
                df.to_excel(fpath, index=False, engine="openpyxl")
                renamed_files.append(fname_nfc)
                print(f"  [이름변경] {fname_nfc}: R예외_아웃 → 예외_아웃")
                continue

            if 'order' not in df.columns or 'batter' not in df.columns:
                print(f"  [경고] {fname_nfc}: order/batter 컬럼 없음, 스킵")
                error_files.append(fname_nfc)
                continue

            order_idx = df.columns.get_loc('order')
            df.insert(order_idx + 1, '예외_아웃', '')

            df.to_excel(fpath, index=False, engine="openpyxl")
            added_files.append(fname_nfc)
            print(f"  [추가] {fname_nfc}")

        except Exception as e:
            print(f"  [오류] {fname_nfc} 처리 실패: {e}")
            error_files.append(fname_nfc)

    print("\n" + "=" * 60)
    print(f"예외_아웃 열 백필 완료")
    print(f"신규 추가: {len(added_files)} / 이름변경: {len(renamed_files)} / 이미 존재(스킵): {len(skipped_files)} / 오류: {len(error_files)}")
    if error_files:
        print(f"오류 파일: {error_files}")
    print("=" * 60)


# ==================== 실행 ====================
#   전체 실행 (기본 YEAR)      : backfill_예외_아웃_column()
#   특정 파일까지만            : backfill_예외_아웃_column(end_file='2022_07_a')
#   특정 파일부터              : backfill_예외_아웃_column(start_file='2022_04_a')
#   특정 구간만                : backfill_예외_아웃_column(start_file='2022_04_a', end_file='2022_07_c')
#   다른 연도 전체             : backfill_예외_아웃_column(year=2023)
#   연도를 안 적어도 파일명에서 자동 인식 : backfill_예외_아웃_column(start_file='2023_04_a', end_file='2023_07_c')
backfill_예외_아웃_column(2026)

# 각 파일에 RBI 추가

In [ ]:
import os
import re
import unicodedata
import pandas as pd


# ==================== 설정 ====================
YEAR = 2023
DRIVE_BASE = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"


# ── 파일 범위 지정 헬퍼 (3. 주자상황 체크.ipynb 와 동일한 규칙)
#   특정 파일까지만  : end_file='2022_07_a'
#   특정 파일부터    : start_file='2022_04_a'
#   특정 구간만      : start_file='2022_04_a', end_file='2022_07_c'
#   year 생략 시 start_file/end_file 앞 4자리(예: '2023_04_a' -> 2023)로 자동 판단
def _slice_file_range(source_files, start_file=None, end_file=None):
    """정렬된 source_files를 file stem 기준 [start_file, end_file] 구간만 남긴다.
    start_file 생략 -> 처음부터 / end_file 생략 -> 끝까지 (양끝 포함).
    '2022_07_a' / '2022_07_a.xlsx' 둘 다 허용."""
    if not source_files or (not start_file and not end_file):
        return source_files
    stems = [os.path.splitext(os.path.basename(p))[0] for p in source_files]

    def _idx(name, label):
        stem = os.path.splitext(name)[0]
        if stem not in stems:
            raise ValueError(f"{label}='{name}' 을(를) 파일 목록에서 찾을 수 없습니다")
        return stems.index(stem)

    i = _idx(start_file, 'start_file') if start_file else 0
    j = _idx(end_file, 'end_file') if end_file else len(stems) - 1
    if i > j:
        raise ValueError(f"범위 오류: start_file='{stems[i]}' 이 end_file='{stems[j]}' 보다 뒤입니다")
    return source_files[i:j + 1]


def _infer_year(start_file, end_file, fallback_year):
    """start_file/end_file 앞 4자리 숫자(YYYY_MM_x 형식)를 연도로 추정한다.
    둘 다 없거나 형식이 안 맞으면 fallback_year를 그대로 쓴다."""
    for name in (start_file, end_file):
        if name:
            m = re.match(r'^(\d{4})_', os.path.basename(name))
            if m:
                return int(m.group(1))
    return fallback_year


def _collect_pbp_files(year):
    """{DRIVE_BASE}/{year} 하위 'N월' 폴더들을 순서대로 훑어 PBP 파일 경로를 정렬된 리스트로 돌려준다."""
    year_dir = f"{DRIVE_BASE}/{year}"
    month_pat = re.compile(r'^\d+월$')
    file_pat  = re.compile(rf'^{year}_\d{{2}}_[a-z]+\.xlsx$')
    source_files = []
    for entry in sorted(os.listdir(year_dir)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(year_dir, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if file_pat.match(f_nfc):
                source_files.append(os.path.join(month_dir, f_nfc))
    source_files.sort()
    return source_files


# ==================== RBI 열 일괄 백필 ====================
def backfill_RBI_column(year=None, start_file=None, end_file=None):
    """
    {DRIVE_BASE}/{year} 하위 월별 폴더(예: 3월/, 4월/...)의 PBP 엑셀 파일(YYYY_MM_x.xlsx) 중
    start_file~end_file 구간만(둘 다 생략 시 전체) 스캔해서 home_score와 position 사이에 RBI 열이 있도록 맞춘다.
      - year를 생략하면 start_file/end_file에 적힌 연도(앞 4자리)를 그대로 쓰고,
        둘 다 없으면 위 YEAR 기본값을 쓴다.
      - 이미 'RBI' 열이 있으면 스킵
      - 없으면 home_score 바로 다음에 빈 칸으로 새로 추가
    안전하게 여러 번 실행 가능.
    """
    year = year if year is not None else _infer_year(start_file, end_file, YEAR)
    year_dir = f"{DRIVE_BASE}/{year}"
    if not os.path.exists(year_dir):
        print(f"경로가 존재하지 않습니다: {year_dir}")
        return

    source_files = _collect_pbp_files(year)
    _n_all = len(source_files)
    source_files = _slice_file_range(source_files, start_file, end_file)
    if not source_files:
        print("해당 범위에 파일이 없습니다.")
        return
    if len(source_files) != _n_all:
        _s = os.path.splitext(os.path.basename(source_files[0]))[0]
        _e = os.path.splitext(os.path.basename(source_files[-1]))[0]
        print(f"[{year}년 / 범위 지정] {_s} ~ {_e} ({len(source_files)}/{_n_all}개 파일)\n")
    else:
        print(f"[{year}년 / 전체] {_n_all}개 파일\n")

    added_files = []
    skipped_files = []
    error_files = []

    for fpath in source_files:
        fname_nfc = os.path.basename(fpath)
        try:
            df = pd.read_excel(fpath, engine="openpyxl")

            if 'RBI' in df.columns:
                skipped_files.append(fname_nfc)
                print(f"  [스킵] {fname_nfc}: 이미 열이 존재함")
                continue

            if 'home_score' not in df.columns or 'position' not in df.columns:
                print(f"  [경고] {fname_nfc}: home_score/position 컬럼 없음, 스킵")
                error_files.append(fname_nfc)
                continue

            home_score_idx = df.columns.get_loc('home_score')
            df.insert(home_score_idx + 1, 'RBI', '')

            df.to_excel(fpath, index=False, engine="openpyxl")
            added_files.append(fname_nfc)
            print(f"  [추가] {fname_nfc}")

        except Exception as e:
            print(f"  [오류] {fname_nfc} 처리 실패: {e}")
            error_files.append(fname_nfc)

    print("\n" + "=" * 60)
    print(f"RBI 열 백필 완료")
    print(f"신규 추가: {len(added_files)} / 이미 존재(스킵): {len(skipped_files)} / 오류: {len(error_files)}")
    if error_files:
        print(f"오류 파일: {error_files}")
    print("=" * 60)


# ==================== 실행 ====================
#   전체 실행 (기본 YEAR)      : backfill_RBI_column()
#   특정 파일까지만            : backfill_RBI_column(end_file='2022_07_a')
#   특정 파일부터              : backfill_RBI_column(start_file='2022_04_a')
#   특정 구간만                : backfill_RBI_column(start_file='2022_04_a', end_file='2022_07_c')
#   다른 연도 전체             : backfill_RBI_column(year=2022)
#   연도를 안 적어도 파일명에서 자동 인식 : backfill_RBI_column(start_file='2022_04_a', end_file='2022_07_c')
backfill_RBI_column(2026)

# 특정 열 숨기기

In [ ]:
import os
import re
import unicodedata
from openpyxl import load_workbook


# ==================== 설정 ====================
YEAR = 2023
DRIVE_BASE = "/content/drive/MyDrive/4-1/졸업작품 (고교야구 기록지)"

# 숨길 열 목록
HIDE_COLUMNS = [
    'position', 'tournament', 'details', 'year', 'date', 'time', 'stadium',
    'team1', 'score1', 'result1', 'team2', 'score2', 'result2',
]


# ── 파일 범위 지정 헬퍼 (3. 주자상황 체크.ipynb 와 동일한 규칙)
#   특정 파일까지만  : end_file='2022_07_a'
#   특정 파일부터    : start_file='2022_04_a'
#   특정 구간만      : start_file='2022_04_a', end_file='2022_07_c'
#   year 생략 시 start_file/end_file 앞 4자리(예: '2023_04_a' -> 2023)로 자동 판단
def _slice_file_range(source_files, start_file=None, end_file=None):
    """정렬된 source_files를 file stem 기준 [start_file, end_file] 구간만 남긴다.
    start_file 생략 -> 처음부터 / end_file 생략 -> 끝까지 (양끝 포함).
    '2022_07_a' / '2022_07_a.xlsx' 둘 다 허용."""
    if not source_files or (not start_file and not end_file):
        return source_files
    stems = [os.path.splitext(os.path.basename(p))[0] for p in source_files]

    def _idx(name, label):
        stem = os.path.splitext(name)[0]
        if stem not in stems:
            raise ValueError(f"{label}='{name}' 을(를) 파일 목록에서 찾을 수 없습니다")
        return stems.index(stem)

    i = _idx(start_file, 'start_file') if start_file else 0
    j = _idx(end_file, 'end_file') if end_file else len(stems) - 1
    if i > j:
        raise ValueError(f"범위 오류: start_file='{stems[i]}' 이 end_file='{stems[j]}' 보다 뒤입니다")
    return source_files[i:j + 1]


def _infer_year(start_file, end_file, fallback_year):
    """start_file/end_file 앞 4자리 숫자(YYYY_MM_x 형식)를 연도로 추정한다.
    둘 다 없거나 형식이 안 맞으면 fallback_year를 그대로 쓴다."""
    for name in (start_file, end_file):
        if name:
            m = re.match(r'^(\d{4})_', os.path.basename(name))
            if m:
                return int(m.group(1))
    return fallback_year


def _collect_pbp_files(year):
    """{DRIVE_BASE}/{year} 하위 'N월' 폴더들을 순서대로 훑어 PBP 파일 경로를 정렬된 리스트로 돌려준다."""
    year_dir = f"{DRIVE_BASE}/{year}"
    month_pat = re.compile(r'^\d+월$')
    file_pat  = re.compile(rf'^{year}_\d{{2}}_[a-z]+\.xlsx$')
    source_files = []
    for entry in sorted(os.listdir(year_dir)):
        entry_nfc = unicodedata.normalize('NFC', entry)
        if not month_pat.match(entry_nfc):
            continue
        month_dir = os.path.join(year_dir, entry_nfc)
        if not os.path.isdir(month_dir):
            continue
        for f in sorted(os.listdir(month_dir)):
            f_nfc = unicodedata.normalize('NFC', f)
            if file_pat.match(f_nfc):
                source_files.append(os.path.join(month_dir, f_nfc))
    source_files.sort()
    return source_files


# ==================== 특정 열 숨기기 일괄 처리 ====================
def hide_columns(year=None, start_file=None, end_file=None, hide_columns=HIDE_COLUMNS):
    """
    {DRIVE_BASE}/{year} 하위 월별 폴더(예: 3월/, 4월/...)의 PBP 엑셀 파일(YYYY_MM_x.xlsx) 중
    start_file~end_file 구간만(둘 다 생략 시 전체) 스캔해서 hide_columns 에 적힌 열을
    엑셀에서 '숨김' 처리한다. (데이터는 그대로 두고 화면에서만 접힘)
      - year를 생략하면 start_file/end_file에 적힌 연도(앞 4자리)를 그대로 쓰고,
        둘 다 없으면 위 YEAR 기본값을 쓴다.
      - 헤더(1행)에서 열 이름을 찾아 해당 열만 숨김 처리
      - 이미 숨겨져 있어도 그대로 유지 (안전하게 여러 번 실행 가능)
    """
    year = year if year is not None else _infer_year(start_file, end_file, YEAR)
    year_dir = f"{DRIVE_BASE}/{year}"
    if not os.path.exists(year_dir):
        print(f"경로가 존재하지 않습니다: {year_dir}")
        return

    source_files = _collect_pbp_files(year)
    _n_all = len(source_files)
    source_files = _slice_file_range(source_files, start_file, end_file)
    if not source_files:
        print("해당 범위에 파일이 없습니다.")
        return
    if len(source_files) != _n_all:
        _s = os.path.splitext(os.path.basename(source_files[0]))[0]
        _e = os.path.splitext(os.path.basename(source_files[-1]))[0]
        print(f"[{year}년 / 범위 지정] {_s} ~ {_e} ({len(source_files)}/{_n_all}개 파일)\n")
    else:
        print(f"[{year}년 / 전체] {_n_all}개 파일\n")

    hide_set = set(hide_columns)
    done_files = []
    error_files = []

    for fpath in source_files:
        fname_nfc = os.path.basename(fpath)
        try:
            wb = load_workbook(fpath)
            ws = wb.active

            # 헤더(1행)에서 열 이름 -> 열 문자(A, B, ...) 매핑
            header = {}
            for cell in ws[1]:
                if cell.value is not None:
                    header[str(cell.value)] = cell.column_letter

            hidden_now = []
            missing = []
            for col_name in hide_columns:
                if col_name in header:
                    ws.column_dimensions[header[col_name]].hidden = True
                    hidden_now.append(col_name)
                else:
                    missing.append(col_name)

            wb.save(fpath)
            done_files.append(fname_nfc)
            msg = f"  [숨김] {fname_nfc}: {len(hidden_now)}개 열"
            if missing:
                msg += f" (없는 열: {missing})"
            print(msg)

        except Exception as e:
            print(f"  [오류] {fname_nfc} 처리 실패: {e}")
            error_files.append(fname_nfc)

    print("\n" + "=" * 60)
    print(f"열 숨기기 완료")
    print(f"처리: {len(done_files)} / 오류: {len(error_files)}")
    if error_files:
        print(f"오류 파일: {error_files}")
    print("=" * 60)


# ==================== 실행 ====================
#   전체 실행 (기본 YEAR)      : hide_columns()
#   특정 파일까지만            : hide_columns(end_file='2022_07_a')
#   특정 파일부터              : hide_columns(start_file='2022_04_a')
#   특정 구간만                : hide_columns(start_file='2022_04_a', end_file='2022_07_c')
#   다른 연도 전체             : hide_columns(year=2022)
#   연도를 안 적어도 파일명에서 자동 인식 : hide_columns(start_file='2022_04_a', end_file='2022_07_c')
hide_columns(2026)